# Training model kemacetan (fine-tune) v3: tambah frame malam
Di Google Drive `MyDrive/cctv/` harus ada `runs/v2/weights/best.pt`, `dataset_final` (folder atau `dataset_final.zip`); frame malam diunduh otomatis dari Roboflow. Runtime: **GPU** (Runtime > Change runtime type).

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
import os, shutil, glob
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/cctv'
DS = '/content/dataset_final'

shutil.copy(f'{BASE}/runs/v2/weights/best.pt', '/content/best.pt')  # mulai dari model v2
# dataset disalin ke disk lokal Colab (jauh lebih cepat daripada membaca langsung dari Drive)
if os.path.isdir(f'{BASE}/dataset_final'):
    if not os.path.isdir(DS):
        shutil.copytree(f'{BASE}/dataset_final', DS)
elif os.path.exists(f'{BASE}/dataset_final.zip'):
    !unzip -q -o {BASE}/dataset_final.zip -d /content
else:
    raise SystemExit('dataset_final tidak ditemukan di Drive')

# frame malam Bogor: unduh langsung dari Roboflow (project CCTV Malam v1, label auto-label)
RF_URL = 'ISI_LINK_EKSPOR_ROBOFLOW'  # Roboflow > CCTV Malam > Versions > v1 > Download (YOLOv8) > "show download code" > link
!curl -sL "{RF_URL}" -o /content/malam_rf.zip
!unzip -q -o /content/malam_rf.zip -d /content/malam_rf
remap = {0: 2, 1: 2, 2: 0, 3: 3}  # Roboflow: bus,car,motorcycle,truck -> model: motorcycle,bus,car,truck; bus malam = angkot -> car
imgs = sorted(glob.glob('/content/malam_rf/train/images/*.jpg'))
val = imgs[-30:]  # 30 frame terakhir (21:00-21:10) tidak ikut training, untuk evaluasi malam
for im in imgs:
    s = 'val' if im in val else 'train'
    name = 'malam_' + os.path.basename(im)
    shutil.copy(im, f'{DS}/images/{s}/{name}')
    lb = im.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    lines = [' '.join([str(remap[int(l.split()[0])])] + l.split()[1:]) for l in open(lb) if l.strip()]
    open(f'{DS}/labels/{s}/' + name.rsplit('.', 1)[0] + '.txt', 'w').write('\n'.join(lines))
open(f'{DS}/val_malam.txt', 'w').write('\n'.join('./images/val/malam_' + os.path.basename(p) for p in val))
open(f'{DS}/malam.yaml', 'w').write(open(f'{DS}/bogor.yaml').read().replace('val_bogor.txt', 'val_malam.txt'))

# pemeriksaan: harus 2685 train, 132 val, 72 test
for s in ('train', 'val', 'test'):
    print(s, len(glob.glob(f'{DS}/images/{s}/*')), 'gambar', len(glob.glob(f'{DS}/labels/{s}/*.txt')), 'label')
for f in ('data.yaml', 'bogor.yaml', 'val_bogor.txt', 'malam.yaml', 'val_malam.txt'):
    assert os.path.exists(f'{DS}/{f}'), f + ' tidak ada'
print(open(f'{DS}/data.yaml').read())

## Training
Hasil disimpan ke Drive supaya tidak hilang jika sesi terputus.

In [ ]:
from ultralytics import YOLO
model = YOLO('/content/best.pt')
model.train(
    data=f'{DS}/data.yaml',
    imgsz=640, epochs=40, patience=10, batch=16, workers=2,
    project=f'{BASE}/runs', name='v3', exist_ok=True,
)

## Evaluasi v2 vs v3: frame malam (30 gambar, blok waktu yang tidak ikut training) dan frame siang Bogor (14 gambar)

In [ ]:
RUN = f'{BASE}/runs/v3'
for data in ('malam.yaml', 'bogor.yaml'):
    for name, w in [('v2', '/content/best.pt'), ('v3', RUN + '/weights/best.pt')]:
        m = YOLO(w).val(data=f'{DS}/{data}', imgsz=640, plots=False)
        print(data, name, 'P=%.3f R=%.3f mAP50=%.3f mAP50-95=%.3f' % (m.box.mp, m.box.mr, m.box.map50, m.box.map))

## Ekspor untuk worker (OpenVINO, ukuran 640)

In [ ]:
best = YOLO(RUN + '/weights/best.pt')
best.export(format='openvino', imgsz=640)
!cp -r {RUN}/weights/best_openvino_model {BASE}/best_openvino_model_v3
!ls {BASE}